In [1]:
%pip install -q Flask numpy opencv-contrib-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 MB 12.0 MB/s eta 0:00:00


In [ ]:
from __future__ import annotations
import base64
import binascii
import csv
import os
import re
import threading
from datetime import datetime
from pathlib import Path
import cv2
import numpy as np
from flask import Flask, jsonify, render_template_string, request, send_file
import requests # Import requests for downloading files

# All application data is stored in the Colab runtime. Download the CSV before
# the runtime disconnects, or mount Google Drive and change DATA_DIR below.
DATA_DIR = Path(os.environ.get("ATTENDANCE_DATA_DIR", "/content/attendance_data"))
PEOPLE_DIR = DATA_DIR / "people"
CSV_PATH = DATA_DIR / "attendance.csv"
IMAGE_SIZE = (200, 200)
FACE_THRESHOLD = 70.0  # LBPH distance: lower means a closer match.

DATA_DIR.mkdir(parents=True, exist_ok=True);
PEOPLE_DIR.mkdir(parents=True, exist_ok=True)

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 12 * 1024 * 1024
state_lock = threading.RLock()

# OpenCV's contrib package includes the cv2.face module required for LBPH.
if not hasattr(cv2, "face") or not hasattr(cv2.face, "LBPHFaceRecognizer_create"):
    raise RuntimeError(
        "OpenCV contrib is required. In Colab run: !pip install -q opencv-contrib\npython-headless flask, then restart the runtime."
    )

# --- Start of modified Haar cascade loading logic ---
HAARCASCADE_URL = "https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/haarcascade_frontalface_default.xml"
HAASCASCADE_FILENAME = "haarcascade_frontalface_default.xml"
DOWNLOAD_DIR = Path("/content/") # Use Path for easier manipulation

# Ensure the download directory exists
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
face_cascade_filepath = DOWNLOAD_DIR / HAASCASCADE_FILENAME

if not face_cascade_filepath.exists():
    print(f"Downloading {HAASCASCADE_FILENAME}...")
    try:
        response = requests.get(HAARCASCADE_URL)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        face_cascade_filepath.write_bytes(response.content)
        print(f"Downloaded {HAASCASCADE_FILENAME} to {face_cascade_filepath}")
    except Exception as e:
        print(f"Error downloading {HAASCASCADE_FILENAME}: {e}")
        raise RuntimeError(f"Failed to download {HAASCASCADE_FILENAME}")

# Set FACE_CASCADE_PATH environment variable to the downloaded file's path
os.environ["FACE_CASCADE_PATH"] = face_cascade_filepath.as_posix()

# Now, initialize cascade_path using the environment variable or cv2.data.haarcascades
# The environment variable should now point to a valid, existing file.
cascade_path = Path(
    os.environ.get(
        "FACE_CASCADE_PATH",
        str(Path(cv2.data.haarcascades) / HAASCASCADE_FILENAME), # Use HAASCASCADE_FILENAME constant
    )
)

# This check should now always be true because we downloaded it or it existed.
if not cascade_path.exists():
    raise RuntimeError(
        f"Could not load the Haar cascade. The file '{HAASCASCADE_FILENAME}' "
        f"was not found at '{cascade_path}' even after attempting download. "
        f"Please check if the file exists or if the download failed."
    )
# --- End of modified Haar cascade loading logic ---

face_detector = cv2.CascadeClassifier(str(cascade_path))
if face_detector.empty():
    raise RuntimeError(
        "Could not load the Haar cascade. Set FACE_CASCADE_PATH to "
        "haarcascade_frontalface_default.xml or place that file beside app.py."
    )
recognizer = cv2.face.LBPHFaceRecognizer_create()
label_to_name: dict[int, str] = {}
model_ready = False

def ensure_csv() -> None:
    if not CSV_PATH.exists():
        with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["Date", "Time", "Name", "Status"])

def safe_person_id(name: str) -> str:
    """Make a portable directory name while keeping display names unchanged."""
    slug = re.sub(r"[^a-zA-Z0-9_-]+", "_", name.strip()).strip("_-_")
    return (slug or "person").lower()[:48]

def decode_data_url(data_url: str) -> np.ndarray:
    if not isinstance(data_url, str) or "," not in data_url:
        raise ValueError("Image was not sent in a valid format.")
    encoded = data_url.split(",", 1)[1]
    raw = base64.b64decode(encoded, validate=True)
    image = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError("Could not decode the captured image.")
    return image

def largest_face(gray: np.ndarray) -> tuple[int, int, int, int] | None:
    faces = face_detector.detectMultiScale(
        gray, scaleFactor=1.15, minNeighbors=5, minSize=(60, 60)
    )
    if len(faces) == 0:
        return None
    return max(faces, key=lambda box: int(box[2]) * int(box[3]))

def train_model() -> int:
    """Train LBPH from saved face crops. Must be called under state_lock."""
    global label_to_name, model_ready, recognizer
    samples: list[np.ndarray] = []

    labels: list[int] = []
    new_label_to_name: dict[int, str] = {}
    person_dirs = sorted(p for p in PEOPLE_DIR.iterdir() if p.is_dir())
    for label, person_dir in enumerate(person_dirs):
        name_file = person_dir / "display_name.txt"
        display_name = (
            name_file.read_text(encoding="utf-8").strip()
            if name_file.exists()
            else person_dir.name
        )
        new_label_to_name[label] = display_name
        for image_path in sorted(person_dir.glob("*.png")):
            face = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if face is not None:
                samples.append(cv2.resize(face, IMAGE_SIZE))
                labels.append(label)
    if not samples:
        label_to_name = {}
        model_ready = False
        return 0
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.train(samples, np.asarray(labels, dtype=np.int32))
    label_to_name = new_label_to_name
    model_ready = True
    return len(samples)

def mark_attendance(name: str) -> bool:
    """Write at most one attendance row per person per local calendar day."""
    now = datetime.now()
    date_string = now.strftime("%Y-%m-%d")
    ensure_csv()
    with state_lock:
        with CSV_PATH.open("r", newline="", encoding="utf-8") as f:
            already_marked = any(
                row.get("Date") == date_string and row.get("Name", "").casefold()
                == name.casefold()
                for row in csv.DictReader(f)
            )
        if already_marked:
            return False
        with CSV_PATH.open("a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([date_string, now.strftime("%H:%M:%S"), name,
"Present"])
    return True

ensure_csv()
with state_lock:
    train_model()

@app.get("/")
def home():
    return render_template_string(PAGE)

@app.post("/api/register")
def register_person():
    payload = request.get_json(silent=True) or {}
    name = str(payload.get("name", "")).strip()
    images = payload.get("images", [])
    if not name or len(name) > 60:
        return jsonify(error="Enter a name between 1 and 60 characters."), 400
    if not isinstance(images, list) or not images:
        return jsonify(error="No camera frames were received. Allow camera access \nand try again."), 400
    # Display names are stored separately from their safe folder names.
    person_id = safe_person_id(name)
    person_dir = PEOPLE_DIR / person_id
    person_dir.mkdir(parents=True, exist_ok=True)
    (person_dir / "display_name.txt").write_text(name, encoding="utf-8")
    saved = 0
    for image_data in images[:20]:
        try:
            frame = decode_data_url(image_data)
        except (ValueError, binascii.Error):
            continue
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        box = largest_face(gray)
        if box is None:
            continue
        x, y, w, h = box
        margin_x, margin_y = int(w * 0.12), int(h * 0.12)

        x1, y1 = max(0, x - margin_x), max(0, y - margin_y)
        x2, y2 = min(gray.shape[1], x + w + margin_x), min(gray.shape[0], y + h +
margin_y)
        face = cv2.resize(gray[y1:y2, x1:x2], IMAGE_SIZE)
        face = cv2.equalizeHist(face)
        filename = person_dir / \
            f"face_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}_{saved:02d}.png"
        cv2.imwrite(str(filename), face)
        saved += 1
    if not saved:
        return jsonify(error="No face was detected in the captured frames. Face the \ncamera with good lighting and try again."), 400
    with state_lock:
        sample_count = train_model()
    return jsonify(message=f"Registered {name} using {saved} face images.",
                   samples=saved, total_samples=sample_count)

@app.post("/api/recognize")
def recognize_frame():
    payload = request.get_json(silent=True) or {}
    try:
        frame = decode_data_url(payload.get("image", ""))
    except (ValueError, binascii.Error) as exc:
        return jsonify(error=str(exc)), 400
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    boxes = face_detector.detectMultiScale(gray, scaleFactor=1.15, minNeighbors=5,
                                           minSize=(60, 60))
    results = []
    for x, y, w, h in boxes:
        crop = cv2.resize(gray[y:y + h, x:x + w], IMAGE_SIZE)
        crop = cv2.equalizeHist(crop)
        name, confidence = "Unknown", None
        with state_lock:
            if model_ready:
                predicted_label, distance = recognizer.predict(crop)
                confidence = round(float(distance), 1)
                if distance < FACE_THRESHOLD and predicted_label in label_to_name:
                    name = label_to_name[predicted_label]
                newly_marked = mark_attendance(name) if name != "Unknown" else False

            results.append({
                "box": [int(x), int(y), int(w), int(h)],
                "name": name,
                "distance": confidence,
                "attendance": "Marked present" if newly_marked else ("Already\nmarked today" if name != "Unknown" else "Not recognized"),
            })
    return jsonify(faces=results, model_ready=model_ready)

@app.get("/api/attendance")
def attendance_rows():
    ensure_csv()
    with CSV_PATH.open("r", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return jsonify(rows=rows[-100:][::-1])

@app.get("/attendance.csv")
def download_csv():
    ensure_csv()
    return send_file(CSV_PATH, as_attachment=True, download_name="attendance.csv",
                     mimetype="text/csv")

PAGE = r"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width,initial-scale=1">
  <title>Face Attendance</title>
  <style>
    :root { font-family: Inter, system-ui, Arial, sans-serif; color: #172033;
background: #f3f6fb; }
* { box-sizing: border-box; }
    body { margin: 0; }
.wrap { max-width: 1050px; margin: 32px auto; padding: 0 18px; }
.hero { margin-bottom: 20px; }
.hero h1 { margin: 0 0 6px; font-size: 30px; }
.hero p { margin: 0; color: #5c6980; }
.grid { display: grid; grid-template-columns: minmax(0, 1.5fr) minmax(270px,
1fr); gap: 18px; }
.card { background: white; border: 1px solid #e1e7f0; border-radius: 16px;

padding: 18px; box-shadow: 0 8px 24px #182f5510; }
.camera { position: relative; background: #111; border-radius: 12px; overflow:
hidden; min-height: 260px; aspect-ratio: 4/3; }
.camera video { width: 100%; height: 100%; object-fit: cover; display: block;
transform: scaleX(-1); }
    #overlay { position: absolute; inset: 0; width: 100%; height: 100%; pointer
events: none; }
.status { padding: 10px 12px; background: #f3f6fb; border-radius: 9px; margin:
12px 0; color: #47546b; font-size: 14px; min-height: 40px; } /* Added min-height */
    label { display: block; font-size: 13px; font-weight: 650; margin: 12px 0
6px; }
    input { width: 100%; padding: 11px 12px; border: 1px solid #cdd6e3; border
radius: 9px; font: inherit; }
    button, .link { display: inline-block; border: 0; border-radius: 9px; padding:
10px 13px; font-weight: 650; font: inherit; cursor: pointer; text-decoration:
none; }
.primary { background: #245fe8; color: white; }
.secondary { background: #e9efff; color: #1948b3; }
.link { background: #edf2f7; color: #233249; margin-top: 12px; }
.row { display: flex; gap: 8px; flex-wrap: wrap; }
.hint { font-size: 12px; color: #637087; line-height: 1.45; }
.tablebox { overflow: auto; max-height: 250px; margin-top: 10px; }
    table { width: 100%; border-collapse: collapse; font-size: 13px; }
    th, td { text-align: left; padding: 8px; border-bottom: 1px solid #edf0f5;
white-space: nowrap; }
    th { color: #647086; }
.warning { font-size: 12px; color: #745518; background: #fff7df; padding: 10px;
border-radius: 9px; margin-top: 14px; }
    @media (max-width: 760px) {
.grid { grid-template-columns: 1fr; }
.wrap { margin: 20px auto; }
.hero h1 { font-size: 25px; }
    }
  </style>
</head>
<body>
  <main class="wrap">
    <header class="hero">
      <h1>Real-Time Face Attendance</h1>
      <p>OpenCV face detection · LBPH face matching · Flask dashboard</p>
    </header>
    <div class="grid">

      <section class="card">
        <div class="camera">
          <video id="video" autoplay muted playsinline></video>
          <canvas id="overlay"></canvas>
        </div>
        <div id="status" class="status">Click “Start camera”. The browser will
ask for camera permission; choose Allow.</div>
        <div class="row">
          <button class="primary" id="startCamera">Start camera</button>
          <button class="secondary" id="stopCamera" disabled>Stop camera</button>
        </div>
        <label for="name">Register a person</label>
        <input id="name" maxlength="60" placeholder="Enter their name">
        <p class="hint">Keep your face visible and move slightly while 8 frames are
captured.
          Ask for consent before collecting face images.</p>
        <button class="primary" id="register" disabled>Capture &amp; register</button>
        <div class="warning">Demo only: LBPH can misidentify people and can be
spoofed by a
          photo. Do not use as the sole basis for consequential attendance or
decisions.</div>

        <div class="card" style="margin-top: 18px;">
            <h3 style="margin: 0 0 10px;">Camera Debug Panel</h3>
            <p class="hint">This panel shows real-time camera status for debugging.</p>
            <table style="width: 100%;">
                <tbody>
                    <tr><th>Browser:</th><td id="debug-browser">N/A</td></tr>
                    <tr><th>HTTPS:</th><td id="debug-https">N/A</td></tr>
                    <tr><th>Secure Context:</th><td id="debug-secure-context">N/A</td></tr>
                    <tr><th>getUserMedia:</th><td id="debug-getusermedia">N/A</td></tr>
                    <tr><th>Camera Stream:</th><td id="debug-stream">Off</td></tr>
                    <tr><th>Video Ready State:</th><td id="debug-ready-state">N/A</td></tr>
                    <tr><th>Video Width:</th><td id="debug-video-width">N/A</td></tr>
                    <tr><th>Video Height:</th><td id="debug-video-height">N/A</td></tr>
                </tbody>
            </table>
        </div>

      </section>
      <section class="card">
        <h2 style="margin: 0 0 6px; font-size: 20px">Today's attendance</h2>
        <p class="hint">The first recognized visit each calendar day is saved as
Present.</p>
        <div class="row">
          <button id="refresh" class="secondary">Refresh list</button>
          <a class="link" href="/attendance.csv">Download CSV</a>
        </div>
        <div class="tablebox">
          <table>
            <thead><tr><th>Date</th><th>Time</th><th>Name</th><th>Status</th></
tr></thead>
            <tbody id="rows"><tr><td colspan="4">No records yet</td></tr></tbody>
          </table>
        </div>
        <p class="hint" id="model">Waiting for camera and trained faces.</p>
      </section>
    </div>
  </main>
  <script>

    const video = document.getElementById('video');
    const overlay = document.getElementById('overlay');
    const ctx = overlay.getContext('2d');
    const statusEl = document.getElementById('status');
    const startCameraButton = document.getElementById('startCamera');
    const stopCameraButton = document.getElementById('stopCamera');
    const nameInput = document.getElementById('name');
    const registerButton = document.getElementById('register');

    let cameraStream = null, timer = null, busy = false;

    function say(text) {
      statusEl.textContent = text;
      console.log('STATUS: ' + text);
    }

    function updateDebugPanel() {
        document.getElementById('debug-browser').textContent = navigator.userAgent;
        document.getElementById('debug-https').textContent = window.location.protocol === 'https:' ? 'Yes' : 'No';
        document.getElementById('debug-secure-context').textContent = window.isSecureContext ? 'Yes' : 'No';
        document.getElementById('debug-getusermedia').textContent = navigator.mediaDevices && navigator.mediaDevices.getUserMedia ? 'Supported' : 'Not Supported';
        document.getElementById('debug-stream').textContent = cameraStream ? 'Active' : 'Inactive';
        document.getElementById('debug-ready-state').textContent = video.readyState !== undefined ? video.readyState : 'N/A';
        document.getElementById('debug-video-width').textContent = video.videoWidth > 0 ? video.videoWidth + 'px' : 'N/A';
        document.getElementById('debug-video-height').textContent = video.videoHeight > 0 ? video.videoHeight + 'px' : 'N/A';
    }

    function updateUIForCameraState() {
      startCameraButton.disabled = cameraStream !== null;
      stopCameraButton.disabled = cameraStream === null;
      registerButton.disabled = cameraStream === null || !nameInput.value.trim();
      updateDebugPanel();
    }

    nameInput.addEventListener('input', () => {
      updateUIForCameraState();
    });

    function showCameraError(error) {
      console.error('CAMERA ERROR:', error);
      let errorMessage = 'An unknown camera error occurred.';
      switch (error.name) {
        case 'NotAllowedError':
        case 'PermissionDeniedError':
          errorMessage = 'Camera permission denied. Please click the camera/lock icon in the browser address bar and allow Camera access, then reload the page.';
          break;
        case 'NotFoundError':
          errorMessage = 'No camera was found on this device.';
          break;
        case 'NotReadableError':
          errorMessage = 'Camera is in use by another application or device. Please close other applications and try again.';
          break;
        case 'OverconstrainedError':
          errorMessage = 'Camera constraints could not be satisfied. Your device may not support the requested video resolution/mode.';
          break;
        case 'SecurityError':
          errorMessage = 'Camera access is blocked by browser security settings. Ensure you are on a secure (HTTPS) connection.';
          break;
        default:
          errorMessage = `Camera error: ${error.message}`;
      }
      say(errorMessage);
      updateUIForCameraState();
    }

    async function startCamera() {
      console.log('START CAMERA CLICKED');
      say('Attempting to start camera...');

      if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia) {
        showCameraError(new Error('getUserMedia is not supported by this browser', { cause: 'BrowserSupport' }));
        return;
      }
      if (!window.isSecureContext) {
        showCameraError(new Error('Camera requires a secure HTTPS context', { cause: 'SecurityError' }));
        return;
      }
      if (cameraStream) { // Camera already running
        say('Camera is already running.');
        return;
      }

      try {
        cameraStream = await navigator.mediaDevices.getUserMedia({
          video: {
            width: { ideal: 640 },
            height: { ideal: 480 },
            facingMode: 'user'
          },
          audio: false
        });
        console.log('CAMERA STREAM RECEIVED');

        video.srcObject = cameraStream;
        video.muted = true;
        video.setAttribute('playsinline', '');

        video.onloadedmetadata = async () => {
          console.log('VIDEO METADATA LOADED');
          updateDebugPanel(); // Update debug panel when metadata is loaded
          try {
            await video.play();
            console.log('VIDEO PLAYING');
            say('Camera started successfully. Face the camera to register or identify.');
            updateUIForCameraState();
            if (timer) clearInterval(timer); // Clear any old timer
            timer = setInterval(scan, 850); // Start scanning for faces
          } catch (error) {
            console.error('VIDEO PLAY ERROR:', error);
            showCameraError(error);
            stopCamera(false); // Ensure camera resources are stopped
          }
        };
        // If metadata is already loaded (e.g., from a previous stream or quick reconnect), trigger it manually
        if (video.readyState >= 2) { // HAVE_CURRENT_DATA or more
            video.onloadedmetadata();
        }

      } catch (error) {
        showCameraError(error);
        stopCamera(false); // Ensure resources are cleaned up on error
      }
    }

    function stopCamera(showMessage = true) {
      console.log('STOP CAMERA CLICKED');
      if (timer) {
        clearInterval(timer);
        timer = null;
      }
      if (cameraStream) {
        cameraStream.getTracks().forEach(track => {
          console.log('Stopping:', track.kind);
          track.stop();
        });
        cameraStream = null;
      }
      video.srcObject = null;
      ctx.clearRect(0, 0, overlay.width, overlay.height);
      if (showMessage) say('Camera stopped.');
      updateUIForCameraState();
    }

    function frameData() {
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth || 640;
      canvas.height = video.videoHeight || 480;
      const context = canvas.getContext('2d');
      context.drawImage(video, 0, 0, canvas.width, canvas.height);
      return canvas.toDataURL('image/jpeg', 0.78);
    }

    async function scan() {
      if (!cameraStream || busy || !video.videoWidth) return;
      busy = true;
      try {
        const response = await fetch('/api/recognize', {
          method: 'POST', headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ image: frameData() })
        });
        const data = await response.json();
        draw(data.faces || []);
        document.getElementById('model').textContent = data.model_ready
          ? 'Recognition model is trained. Lower LBPH distance indicates a closer match.'
          : 'Register at least one person to train the model.';
        if (data.faces?.length) {
          const face = data.faces[0];
          say(face.name === 'Unknown' ? 'Face detected — not recognized.'
            : face.name + ' · ' + face.attendance);
        } else {
          say('No face detected. Move closer and improve lighting.');
        }
        if (data.faces?.some(face => face.name !== 'Unknown'
            && face.attendance === 'Marked present')) await loadRows();
      } catch (error) {
        console.error('Recognition request failed:', error);
        say('Recognition request failed: ' + error.message);
      } finally {
        busy = false;
      }
    }

    function draw(faces) {
      const width = video.videoWidth || 640, height = video.videoHeight || 480;
      overlay.width = width;
      overlay.height = height;
      ctx.clearRect(0, 0, width, height);
      faces.forEach(face => {
        const [x, y, boxWidth, boxHeight] = face.box;
        const mirroredX = width - x - boxWidth;
        ctx.strokeStyle = face.name === 'Unknown' ? '#ffb347' : '#30d69a';
        ctx.lineWidth = 3;
        ctx.strokeRect(mirroredX, y, boxWidth, boxHeight);
        ctx.fillStyle = ctx.strokeStyle;
        ctx.font = 'bold 17px Arial';
        ctx.fillText(face.name, Math.max(0, mirroredX), Math.max(20, y - 7));
      });
    }

    async function register() {
      console.log('Capture & register clicked.');
      const name = nameInput.value.trim();
      if (!name) {
        say('Enter a name first.');
        nameInput.focus();
        return;
      }
      if (!cameraStream) {
        say('Start the camera first to capture images.');
        return;
      }
      registerButton.disabled = true; // Disable button immediately
      say('Capturing 8 frames. Keep your face in view...');
      console.log('Starting image capture for registration.');
      const images = [];
      try {
        for (let index = 0; index < 8; index++) {
          if (video.videoWidth) {
            images.push(frameData());
            console.log(`Captured frame ${index + 1}/8`);
          }
          await new Promise(resolve => setTimeout(resolve, 300));
        }
        const response = await fetch('/api/register', {
          method: 'POST', headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ name, images })
        });
        const data = await response.json();
        if (!response.ok) throw new Error(data.error || 'Registration failed');
        say(data.message + ' Model refreshed.');
        nameInput.value = ''; // Clear name input after successful registration
        console.log(`Registration successful for ${name}. Samples: ${data.samples}, Total: ${data.total_samples}`);
        await loadRows();
      } catch (error) {
        console.error('Registration error:', error);
        say('Registration error: ' + error.message);
      } finally {
        updateUIForCameraState(); // Re-enable register button if camera is running
      }
    }

    async function loadRows() {
      try {
        const response = await fetch('/api/attendance');
        const data = await response.json();
        const body = document.getElementById('rows');
        body.innerHTML = data.rows.length
? data.rows.map(row => `<tr><td>${escapeHtml(row.Date)}</td>`
+ `<td>${escapeHtml(row.Time)}</td><td>${escapeHtml(row.Name)}</td>`
+ `<td>${escapeHtml(row.Status)}</td></tr>`).join('')
          : '<tr><td colspan="4">No records yet</td></tr>';
      } catch (error) { console.error('Failed to load attendance rows:', error); /* Keep the dashboard usable if refresh fails. */ }
    }

    function escapeHtml(value) {
      return String(value).replace(/[&<>"]/g, char => ({
        '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;'
      }[char]));
    }

    // Event Listeners
    startCameraButton.addEventListener('click', startCamera);
    stopCameraButton.addEventListener('click', stopCamera);
    registerButton.addEventListener('click', register);
    document.getElementById('refresh').addEventListener('click', loadRows);

    // Initial setup calls
    updateDebugPanel(); // Populate debug panel initially
    loadRows();
    updateUIForCameraState(); // Set initial button states
  </script>
</body>
</html>"""

def colab_proxy_url(port: int) -> str | None:
    """Return a browser-accessible HTTPS URL when running inside Colab."""
    try:
        from google.colab.output import eval_js
        return str(eval_js(f"google.colab.kernel.proxyPort({port})"))
    except (ImportError, Exception):
        return None

if __name__ == "__main__":
    # Start Flask in the background so Colab can open the secure proxy page.
    # Do not expose Flask's debugger in a notebook/tunnel environment.
    import time
    port = int(os.environ.get("PORT", "8000")) # Changed default port to 8000
    server = threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=port, debug=False,
                                threaded=True, use_reloader=False),
        daemon=True,
    )
    server.start()
    time.sleep(1.5)
    url = colab_proxy_url(port)
    if url:
        print(f"Secure camera URL: {url}", flush=True)
        try:
            from google.colab.output import serve_kernel_port_as_window
            serve_kernel_port_as_window(port)
            print("A browser tab was opened automatically. Click Start camera and "
                  "choose Allow.", flush=True)
        except (ImportError, Exception):
            print("Open the Secure camera URL in a new browser tab, then click "
                  "Start camera and choose Allow.", flush=True)
    else:
        print(f"Open http://127.0.0.1:{port} locally.", flush=True)
    try:
        while server.is_alive():
            time.sleep(1)
    except KeyboardInterrupt:
        pass

Downloaded haarcascade_frontalface_default.xml to /content/haarcascade_frontalface_default.xml
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit


Secure camera URL: https://8000-m-s-kkb-usc1a2-7e0jtd7snmv8-a.us-central1-2.prod.colab.dev
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

A browser tab was opened automatically. Click Start camera and choose Allow.


INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:02] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:02] "GET /api/attendance HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:02] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:13] "POST /api/recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:14] "POST /api/recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:15] "POST /api/recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:16] "POST /api/recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:16] "POST /api/recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:17] "POST /api/recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:18] "POST /api/recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:19] "POST /api/recognize HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Sep/2026 06:36:20